# Faruq-v3 — operational threshold and suppression audit

Audit validation-only pada checkpoint D0 seed 42 yang sudah selesai. Notebook ini menyapu confidence threshold dan membandingkan keluaran native dengan class-agnostic NMS. Tidak ada training dan test tidak diakses.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)


In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_operational_audit.json'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT)
print('TRAINING  : TIDAK')


In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.analysis.faruq_v3_operational_audit',
    '--checkpoint', str(CHECKPOINT),
    '--data-root', str(DATA_ROOT),
    '--output', str(OUTPUT),
    '--split', 'val',
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display

result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['training_executed'] is False
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = pd.DataFrame(result['rows'])
columns = [
    'policy', 'threshold', 'proposal_accessibility',
    'conditional_top1_accuracy', 'correct_decision_precision',
    'correct_decision_recall', 'correct_decision_f1',
    'correct_class_availability', 'ranking_conflict_rate',
    'no_correct_class_candidate_rate', 'multi_label_conflict_rate',
    'mean_predictions_per_image',
]
percentage = {column: '{:.2%}' for column in columns if column not in {'policy', 'threshold', 'mean_predictions_per_image'}}
display(rows[columns].style.format(percentage | {'threshold': '{:.2f}', 'mean_predictions_per_image': '{:.2f}'}))
display(pd.DataFrame([result['baseline'], result['selected']], index=['baseline_native_0.25', 'selected'])[columns].style.format(percentage | {'threshold': '{:.2f}', 'mean_predictions_per_image': '{:.2f}'}))
per_class = pd.DataFrame.from_dict(result['selected_per_class'], orient='index').reset_index(names='class_name')
display(per_class.sort_values('correct_decision_recall').head(10).style.format({'proposal_accessibility': '{:.2%}', 'correct_decision_recall': '{:.2%}'}))
print('PUTUSAN:', result['comparison'])
print('SUMMARY:', OUTPUT)
print('Kirim tabel sweep, baseline-vs-selected, dan 10 kelas terbawah. Jangan training model baru.')
